In [ ]:
import os
import re

import matplotlib.pyplot as plt
import numpy as np
from src.util.data_path import fig_model_error

# Error Comparison Analysis Between ARIMA, LSTM, and Transformer Models

This notebook provides a visualization comparison of error metrics between three AI models:
- ARIMA (Auto-Regressive Integrated Moving Average)
- LSTM (Long Short-Term Memory)
- Transformer

The comparison is made for four different crop types:
- Cassava
- Corn
- Green Bean
- Soybean

The error metrics are obtained from the error_record folder containing evaluation results for each model and crop type.

In [38]:
# Define the base path for error metrics files
base_path = r"C:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\src\model\error_record"

# List of models and crops
models = ["ARIMA", "LSTM", "Transformer"]
crops = ["cassava", "corn", "green_bean", "soybean"]


def parse_error_metrics(file_path):
    """
    Parse error metrics from the file

    Args:
        file_path: Path to the error metrics file

    Returns:
        tuple: (overall_metrics, monthly_metrics)
    """
    with open(file_path, "r") as file:
        content = file.read()

    # Extract overall metrics
    mae_match = re.search(r"Mean Absolute Error \(MAE\): ([\d.]+)", content)
    mape_match = re.search(
        r"Mean Absolute Percentage Error \(MAPE\): ([\d.]+)%", content
    )
    accuracy_match = re.search(r"Model Accuracy: ([\d.]+)%", content)

    overall_metrics = {
        "MAE": float(mae_match.group(1)) if mae_match else None,
        "MAPE": float(mape_match.group(1)) if mape_match else None,
        "Accuracy": float(accuracy_match.group(1)) if accuracy_match else None,
    }

    # Extract monthly metrics
    monthly_data = []
    monthly_section = re.search(
        r"Monthly Evaluation:(.*?)(?:$|\n\n)", content, re.DOTALL
    )

    if monthly_section:
        lines = (
            monthly_section.group(1).strip().split("\n")[2:]
        )  # Skip the header lines
        for line in lines:
            parts = re.split(r"\s+", line.strip())
            if len(parts) >= 5:  # Ensure we have enough parts
                try:
                    month = parts[0] + " " + parts[1]
                    actual = float(parts[2])
                    forecast = float(parts[3])
                    error = float(parts[4])
                    abs_error = float(parts[5])
                    pct_error = float(parts[6])

                    monthly_data.append(
                        {
                            "Month": month,
                            "Actual": actual,
                            "Forecast": forecast,
                            "Error": error,
                            "Abs_Error": abs_error,
                            "Pct_Error": pct_error,
                        }
                    )
                except (ValueError, IndexError):
                    # Skip lines that don't match the expected format
                    continue

    return overall_metrics, monthly_data


# Collect all error metrics
error_data = {}
for crop in crops:
    error_data[crop] = {}
    for model in models:
        file_path = os.path.join(base_path, model, f"{crop}_error_metrics.txt")

        if os.path.exists(file_path):
            error_data[crop][model] = parse_error_metrics(file_path)
        else:
            print(f"File not found: {file_path}")
            error_data[crop][model] = None

## Error Comparison Visualization Functions

Below are functions to create the error comparison visualizations between the three AI models for each crop type.

In [ ]:
def create_error_comparison_plots(crop_name):
    """
    Create error comparison plots between the three models for a specific crop

    Args:
        crop_name: Name of the crop to create plots for
    """
    if crop_name not in error_data or not all(
        error_data[crop_name].get(model) for model in models
    ):
        print(f"Error data for {crop_name} is not complete. Skipping plot creation.")
        return

    # Create a figure with multiple subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"Error Metrics Comparison for {crop_name.capitalize()}", fontsize=20)
    axes = axes.flatten()

    # Define colors for each model for consistent presentation
    model_colors = {"ARIMA": "blue", "LSTM": "green", "Transformer": "red"}

    # 1. Plot Monthly Absolute Errors for all models
    ax = axes[0]
    for model in models:
        _, monthly_data = error_data[crop_name][model]
        months = [item["Month"] for item in monthly_data]
        abs_errors = [item["Abs_Error"] for item in monthly_data]

        # Plot line and markers
        ax.plot(
            range(len(months)),
            abs_errors,
            marker="o",
            linestyle="-",
            color=model_colors[model],
            label=f"{model}",
        )

        # Add model name labels next to each end point
        if len(abs_errors) > 0:
            ax.annotate(
                model,
                (len(months) - 1, abs_errors[-1]),
                textcoords="offset points",
                xytext=(5, 0),
                ha="left",
            )

    ax.set_title("Monthly Absolute Errors")
    ax.set_xlabel("Month")
    ax.set_ylabel("Absolute Error")
    ax.set_xticks(range(len(months)))
    ax.set_xticklabels(months, rotation=45)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend()

    # 2. Plot Monthly Percentage Errors for all models
    ax = axes[1]
    for model in models:
        _, monthly_data = error_data[crop_name][model]
        months = [item["Month"] for item in monthly_data]
        pct_errors = [item["Pct_Error"] for item in monthly_data]

        # Plot line and markers
        ax.plot(
            range(len(months)),
            pct_errors,
            marker="o",
            linestyle="-",
            color=model_colors[model],
            label=f"{model}",
        )

        # Add model name labels next to each end point
        if len(pct_errors) > 0:
            ax.annotate(
                model,
                (len(months) - 1, pct_errors[-1]),
                textcoords="offset points",
                xytext=(5, 0),
                ha="left",
            )

    ax.set_title("Monthly Percentage Errors")
    ax.set_xlabel("Month")
    ax.set_ylabel("Percentage Error (%)")
    ax.set_xticks(range(len(months)))
    ax.set_xticklabels(months, rotation=45)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend()

    # 3. Bar chart for overall metrics (MAE, MAPE)
    ax = axes[2]

    # Extract MAE and MAPE for each model
    metric_values = {model: [] for model in models}
    metric_labels = ["MAE", "MAPE"]

    for model in models:
        overall_metrics, _ = error_data[crop_name][model]
        metric_values[model] = [overall_metrics["MAE"], overall_metrics["MAPE"]]

    x = np.arange(len(metric_labels))
    width = 0.2  # Width of bars
    offsets = [-width, 0, width]  # For positioning bars side by side

    # Plot bars for each model
    for i, model in enumerate(models):
        ax.bar(
            x + offsets[i],
            metric_values[model],
            width,
            label=model,
            color=model_colors[model],
        )

    ax.set_title("Overall Error Metrics")
    ax.set_ylabel("Error Value")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.grid(True, axis="y", linestyle="--", alpha=0.7)
    ax.legend()

    # 4. Bar chart for model accuracy
    ax = axes[3]

    # Extract accuracy for each model
    accuracies = []
    for model in models:
        overall_metrics, _ = error_data[crop_name][model]
        accuracies.append(overall_metrics["Accuracy"])

    # Plot bars
    bars = ax.bar(models, accuracies, color=[model_colors[model] for model in models])

    # Add accuracy values on top of bars
    for bar, accuracy in zip(bars, accuracies):
        ax.annotate(
            f"{accuracy:.2f}%",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),  # 3 points vertical offset
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    ax.set_title("Model Accuracy Comparison")
    ax.set_xlabel("Model")
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)  # Set y-axis from 0 to 100
    ax.grid(True, axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for the title

    # Save the figure
    views_dir = fig_model_error / "views"
    views_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(views_dir / f"{crop_name}_models_comparison.png", dpi=300, bbox_inches="tight")

    return fig

## Generate Error Comparison Plots for Each Crop

Now let's generate the error comparison plots for each crop type: cassava, corn, green_bean, and soybean.

In [40]:
# Create plots for each crop
figures = {}

for crop in crops:
    print(f"Generating error comparison plot for {crop}...")
    figures[crop] = create_error_comparison_plots(crop)
    plt.close()  # Close the figure to free up memory

print("All plots have been generated and saved.")

Generating error comparison plot for cassava...
Generating error comparison plot for corn...
Generating error comparison plot for corn...
Generating error comparison plot for green_bean...
Generating error comparison plot for green_bean...
Generating error comparison plot for soybean...
Generating error comparison plot for soybean...
All plots have been generated and saved.
All plots have been generated and saved.


In [ ]:
# Display the generated comparison images from the saved location
from IPython.display import Image, display

error_img_dir = fig_model_error / "views"

# Check if the directory exists
if not error_img_dir.exists():
    print(f"Error: Directory {error_img_dir} does not exist.")
else:
    # Display each crop comparison image
    for crop in crops:
        img_path = error_img_dir / f"{crop}_models_comparison.png"
        if img_path.exists():
            print(f"\n## Error Comparison for {crop.capitalize()}")
            display(Image(str(img_path)))
        else:
            print(f"Image for {crop} not found at {img_path}")

## Detailed Model Performance Analysis

Let's create a summary table of the key error metrics for all models and crops to facilitate easier comparison:

In [42]:
import pandas as pd
from IPython.display import display

# Create a summary DataFrame of error metrics for all models and crops
summary_data = []

for crop in crops:
    for model in models:
        if error_data[crop] and error_data[crop][model]:
            overall_metrics, _ = error_data[crop][model]
            summary_data.append(
                {
                    "Crop": crop.capitalize(),
                    "Model": model,
                    "MAE": overall_metrics["MAE"],
                    "MAPE (%)": overall_metrics["MAPE"],
                    "Accuracy (%)": overall_metrics["Accuracy"],
                }
            )

# Create the DataFrame
summary_df = pd.DataFrame(summary_data)

# Print the formatted summary table without styling
print("### Summary of Error Metrics Across All Models and Crops")
display(summary_df.round({"MAE": 4, "MAPE (%)": 2, "Accuracy (%)": 2}))

### Summary of Error Metrics Across All Models and Crops


,Crop,Model,MAE,MAPE (%),Accuracy (%)
0,Cassava,ARIMA,0.6833,24.71,77.74
1,Cassava,LSTM,0.1898,6.19,93.82
2,Cassava,Transformer,0.3072,9.32,89.99
3,Corn,ARIMA,0.8603,8.10,92.03
4,Corn,LSTM,0.8723,8.10,91.92
5,Corn,Transformer,1.6953,15.09,84.29
6,Green_bean,ARIMA,1.7079,6.08,93.45
7,Green_bean,LSTM,2.3964,9.02,90.81
8,Green_bean,Transformer,1.6947,6.34,93.50
9,Soybean,ARIMA,1.3168,6.99,92.82


### Best Model Determination

Let's identify the best overall model for each crop type based on the error metrics:

In [43]:
# Determine the best model for each crop
best_models = {}

for crop in crops:
    crop_df = summary_df[summary_df["Crop"] == crop.capitalize()].copy()

    # Normalize metrics for scoring
    # For MAE and MAPE: lower is better (1 - normalized value)
    # For Accuracy: higher is better (normalized value)
    crop_df["MAE_norm"] = 1 - (crop_df["MAE"] - crop_df["MAE"].min()) / (
        crop_df["MAE"].max() - crop_df["MAE"].min() + 1e-10
    )
    crop_df["MAPE_norm"] = 1 - (crop_df["MAPE (%)"] - crop_df["MAPE (%)"].min()) / (
        crop_df["MAPE (%)"].max() - crop_df["MAPE (%)"].min() + 1e-10
    )
    crop_df["Acc_norm"] = (crop_df["Accuracy (%)"] - crop_df["Accuracy (%)"].min()) / (
        crop_df["Accuracy (%)"].max() - crop_df["Accuracy (%)"].min() + 1e-10
    )

    # Calculate combined score (equal weights)
    crop_df["Score"] = (
        crop_df["MAE_norm"] + crop_df["MAPE_norm"] + crop_df["Acc_norm"]
    ) / 3

    # Get the model with the best score
    best_model_row = crop_df.loc[crop_df["Score"].idxmax()]

    best_models[crop] = {
        "model": best_model_row["Model"],
        "mae": best_model_row["MAE"],
        "mape": best_model_row["MAPE (%)"],
        "accuracy": best_model_row["Accuracy (%)"],
        "score": best_model_row["Score"],
    }

# Print the best model for each crop
for crop, info in best_models.items():
    print(f"\nBest model for {crop.capitalize()}: {info['model']}")
    print(f"  - MAE: {info['mae']:.4f}")
    print(f"  - MAPE: {info['mape']:.2f}%")
    print(f"  - Accuracy: {info['accuracy']:.2f}%")
    print(f"  - Overall Score: {info['score']:.4f}")


Best model for Cassava: LSTM
  - MAE: 0.1898
  - MAPE: 6.19%
  - Accuracy: 93.82%
  - Overall Score: 1.0000

Best model for Corn: ARIMA
  - MAE: 0.8603
  - MAPE: 8.10%
  - Accuracy: 92.03%
  - Overall Score: 1.0000

Best model for Green_bean: ARIMA
  - MAE: 1.7079
  - MAPE: 6.08%
  - Accuracy: 93.45%
  - Overall Score: 0.9875

Best model for Soybean: Transformer
  - MAE: 1.0686
  - MAPE: 5.71%
  - Accuracy: 94.18%
  - Overall Score: 1.0000


### Combined Model Performance Visualization

Let's create a single visualization that compares the best metric (Accuracy) across all models and crops:

In [ ]:
# Create a grouped bar chart comparing accuracy across models and crops
plt.figure(figsize=(12, 8))

# Prepare data for plotting
crop_names = [crop.capitalize() for crop in crops]
x = np.arange(len(crop_names))
width = 0.25  # Width of the bars

# Plot bars for each model
for i, model in enumerate(models):
    accuracies = []
    for crop in crops:
        crop_df = summary_df[
            (summary_df["Crop"] == crop.capitalize()) & (summary_df["Model"] == model)
        ]
        accuracies.append(crop_df["Accuracy (%)"].values[0] if not crop_df.empty else 0)

    plt.bar(
        x + (i - 1) * width,
        accuracies,
        width,
        label=model,
        color={"ARIMA": "blue", "LSTM": "green", "Transformer": "red"}[model],
    )

# Add values on top of bars
for i, model in enumerate(models):
    for j, crop in enumerate(crops):
        crop_df = summary_df[
            (summary_df["Crop"] == crop.capitalize()) & (summary_df["Model"] == model)
        ]
        if not crop_df.empty:
            accuracy = crop_df["Accuracy (%)"].values[0]
            plt.text(
                j + (i - 1) * width,
                accuracy + 1,
                f"{accuracy:.1f}%",
                ha="center",
                va="bottom",
                fontsize=9,
            )

# Add chart details
plt.xlabel("Crop Type", fontsize=12, fontweight="bold")
plt.ylabel("Accuracy (%)", fontsize=12, fontweight="bold")
plt.title("Model Accuracy Comparison Across All Crops", fontsize=16, fontweight="bold")
plt.xticks(x, crop_names)
plt.ylim(0, 100)  # Set y-axis from 0 to 100%
plt.legend(title="Model")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Highlight the best model for each crop
for j, crop in enumerate(crops):
    best_model = best_models[crop]["model"]
    i = models.index(best_model)
    accuracy = best_models[crop]["accuracy"]
    plt.plot(j + (i - 1) * width, accuracy, "k*", markersize=10)

plt.tight_layout()
plt.savefig(fig_model_error / "views" / "combined_model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## Conclusion

The error overlap comparison analysis between the three AI models (ARIMA, LSTM, and Transformer) for different crop types reveals several important findings:

1. **Performance Variation by Crop**: Each model performs differently depending on the crop type, suggesting that crop-specific characteristics influence model effectiveness.

2. **Error Patterns**: The monthly error analysis shows how each model's error changes over time, which is crucial for understanding their temporal prediction stability.

3. **Overall Metrics**: Looking at MAE, MAPE, and Accuracy together provides a comprehensive view of model performance beyond any single metric.

4. **Best Model Selection**: The combined score approach helps identify the overall best-performing model for each crop type, taking into account all error metrics.

These insights can guide the selection of the most appropriate model for price forecasting for each crop type, potentially improving agricultural planning and decision-making. The error comparison visualizations serve as a valuable tool for model evaluation and selection in this agricultural forecasting context.